<a href="https://colab.research.google.com/github/arielsharonferdinandus/AirQualityCO_PredictiveAnalytics/blob/data-load/AirQualityCO_PredictiveAnalytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predictive Analytics Group Project: Estimating Carbon Monoxide (CO) from Air Quality Sensors

**Group number / name:** _fill in_  
**Members:** _fill in names_  
**Theme:** Energy and Environment  
**Dataset:** Air Quality UCI (UCI Machine Learning Repository, dataset id 360)

**Notebook structure**
1. Project setup and registration  *(this part)*
2. Data understanding and quality assessment
3. Data preparation
4. Exploratory data analysis
5. Feature engineering
6. Baseline and candidate models
7. Validation and hyperparameter tuning
8. Final evaluation and interpretation
9. Findings and recommendations

---
## 1. Project Setup and Registration

### 1.1 Stakeholder and prediction question

**Stakeholder:** An urban environmental monitoring team that operates a multi-sensor air quality station in an Italian city.

**Problem:** Reference-grade CO analyzers are expensive, need frequent calibration and are often offline, which leaves gaps in the reference record. Low-cost metal-oxide sensors and meteorological probes keep running.

**Prediction question:** *Can we estimate the hourly reference CO concentration, `CO(GT)` (mg/m³), from the same-hour low-cost sensor responses, weather conditions and time-of-day features, accurately enough to fill reference gaps and flag high-CO hours for follow-up?*

### 1.2 Task framing (decision to confirm as a group)

| Item | Choice |
|---|---|
| Prediction type | Regression |
| Target | `CO(GT)`, reference CO concentration in mg/m³ |
| Unit of analysis | One hourly observation |
| Use case | **Same-hour estimation (nowcasting / gap-filling)**, not future forecasting |
| Primary metrics | MAE and RMSE (mg/m³); R² as supporting evidence |
| Validation | Chronological split (train, validation, test) to test temporal generalisation, decided in Part 6/7 |

**Why same-hour estimation:** using contemporaneous sensor readings (`PT08.*`) as predictors is legitimate only if the model is meant to estimate CO *at the same time* the sensors are read. Using `NOx(GT)`, `NO2(GT)` and `C6H6(GT)` is also same-hour, but those are reference-analyzer outputs that may be missing exactly when `CO(GT)` is missing. We will test this in Part 5 and may compare feature sets: (a) low-cost sensors + weather + time only, and (b) adding reference pollutants.

If the group prefers future forecasting instead, the target must be shifted by a defined horizon and every predictor must come from earlier hours. We would change the framing in Parts 5 and 7 and document it.

### 1.3 Registration checklist
- [ ] Register dataset and target at `s.id/listdatapred02`
- [ ] Verify no other group registered the same dataset
- [ ] Confirm it was not used in the individual midterm project
- [ ] Confirm source, publisher, source URL, access date (recorded below from the repository metadata)

### 1.4 Setup: installs, imports, global settings

In [1]:
# Installation (needed in Colab; harmless elsewhere)
!pip install -q ucimlrepo openpyxl

In [2]:
import os
import random
import warnings
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = 'data'                       # relative path, works in Colab and locally
os.makedirs(DATA_DIR, exist_ok=True)
print('Setup complete. Access date:', date.today().isoformat())

Setup complete. Access date: 2026-10-15


### 1.5 Load the data from the original repository

Source: UCI Machine Learning Repository, *Air Quality* (id 360), https://archive.ics.uci.edu/dataset/360/air+quality. We use the `ucimlrepo` package so the notebook downloads the data itself and needs no manual upload.

In [3]:
# Fetch dataset from UCI (needs internet access)
air_quality = fetch_ucirepo(id=360)
ACCESS_DATE = date.today().isoformat()

# Repository metadata
meta = air_quality.metadata
for key in ['uci_id', 'name', 'repository_url', 'data_url', 'abstract', 'area',
            'task', 'num_instances', 'num_features', 'has_missing_values',
            'creators', 'year_of_dataset_creation', 'doi']:
    print(f'{key}: {meta.get(key)}')

uci_id: 360
name: Air Quality
repository_url: https://archive.ics.uci.edu/dataset/360/air+quality
data_url: https://archive.ics.uci.edu/static/public/360/data.csv
abstract: Contains the responses of a gas multisensor device deployed on the field in an Italian city. Hourly responses averages are recorded along with gas concentrations references from a certified analyzer. 
area: Computer Science
task: None
num_instances: 9358
num_features: 15
has_missing_values: no
creators: ['Saverio Vito']
year_of_dataset_creation: 2008
doi: None


In [4]:
# Variable descriptions supplied by the repository (units, roles, missing flags)
air_quality.variables

,name,role,type,demographic,description,units,missing_values
0,Date,Feature,Date,None,None,None,no
1,Time,Feature,Categorical,None,None,None,no
2,CO(GT),Feature,Integer,None,True hourly averaged concentration CO in mg/m^...,mg/m^3,no
3,PT08.S1(CO),Feature,Categorical,None,hourly averaged sensor response (nominally CO...,None,no
4,NMHC(GT),Feature,Integer,None,True hourly averaged overall Non Metanic Hydro...,microg/m^3,no
5,C6H6(GT),Feature,Continuous,None,True hourly averaged Benzene concentration in...,microg/m^3,no
6,PT08.S2(NMHC),Feature,Categorical,None,hourly averaged sensor response (nominally NMH...,None,no
7,NOx(GT),Feature,Integer,None,True hourly averaged NOx concentration in ppb...,ppb,no
8,PT08.S3(NOx),Feature,Categorical,None,hourly averaged sensor response (nominally NOx...,None,no
9,NO2(GT),Feature,Integer,None,True hourly averaged NO2 concentration in micr...,microg/m^3,no


In [5]:
# Build one raw DataFrame. ucimlrepo may return everything in .features
# or split features/targets, so we combine whatever is present.
X = air_quality.data.features
y = air_quality.data.targets
print('features:', None if X is None else X.shape, '| targets:', None if y is None else y.shape)

parts = [p for p in (X, y) if p is not None and p.shape[1] > 0]
df_raw = pd.concat(parts, axis=1)

print('\nCombined raw shape:', df_raw.shape)
df_raw.head()

features: (9357, 15) | targets: None

Combined raw shape: (9357, 15)


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


### 1.6 Verify the data matches the project brief

Expected: 9,357 rows x 15 columns, the 15 named columns, and date coverage 10 March 2004 to 4 April 2005.

In [6]:
expected_cols = ['Date', 'Time', 'CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)',
                 'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)',
                 'PT08.S5(O3)', 'T', 'RH', 'AH']

print('Shape matches (9357, 15):', df_raw.shape == (9357, 15))
print('Missing expected columns:', [c for c in expected_cols if c not in df_raw.columns])
print('Unexpected extra columns:', [c for c in df_raw.columns if c not in expected_cols])

df_raw = df_raw[[c for c in expected_cols if c in df_raw.columns]]   # fixed column order

print('\nData types:')
print(df_raw.dtypes)

print('\nDate range (as stored):', df_raw['Date'].iloc[0], '->', df_raw['Date'].iloc[-1])

Shape matches (9357, 15): True
Missing expected columns: []
Unexpected extra columns: []

Data types:
Date              object
Time              object
CO(GT)           float64
NMHC(GT)           int64
C6H6(GT)         float64
NOx(GT)            int64
NO2(GT)            int64
PT08.S1(CO)        int64
PT08.S2(NMHC)      int64
PT08.S3(NOx)       int64
PT08.S4(NO2)       int64
PT08.S5(O3)        int64
T                float64
RH               float64
AH               float64
dtype: object

Date range (as stored): 3/10/2004 -> 4/4/2005


### 1.7 Save the exact file used for analysis

The submission must include `AirQualityUCI.xlsx`. We write the raw data exactly as downloaded to that file name so the ZIP contains the data the notebook used. No cleaning has been applied at this point (the `-200` handling is done in Part 2/3).

In [7]:
xlsx_path = os.path.join(DATA_DIR, 'AirQualityUCI.xlsx')
df_raw.to_excel(xlsx_path, sheet_name='AirQualityUCI', index=False)
print('Saved:', xlsx_path)

# Reload from the saved file to prove the pipeline reproduces from the file
df = pd.read_excel(xlsx_path, sheet_name='AirQualityUCI')
print('Reloaded shape:', df.shape)

Saved: data/AirQualityUCI.xlsx
Reloaded shape: (9357, 15)


### 1.8 Dataset registration record
Copy these values into the registration form at `s.id/listdatapred02`.

In [8]:
predictors = [c for c in df.columns if c not in ('CO(GT)',)]

registration = {
    'Dataset title': meta.get('name'),
    'Publisher / repository': 'UCI Machine Learning Repository',
    'Creators': meta.get('creators'),
    'Source URL': meta.get('repository_url') or 'https://archive.ics.uci.edu/dataset/360/air+quality',
    'Access date': ACCESS_DATE,
    'Rows x columns': f'{df.shape[0]} x {df.shape[1]}',
    'Date coverage': '10 March 2004 to 4 April 2005 (verify in Part 2)',
    'Prediction type': 'Regression',
    'Target': 'CO(GT)',
    'Candidate predictors': ', '.join(predictors),
}
pd.Series(registration, name='Value').to_frame()

,Value
Dataset title,Air Quality
Publisher / repository,UCI Machine Learning Repository
Creators,[Saverio Vito]
Source URL,https://archive.ics.uci.edu/dataset/360/air+qu...
Access date,2026-10-15
Rows x columns,9357 x 15
Date coverage,10 March 2004 to 4 April 2005 (verify in Part 2)
Prediction type,Regression
Target,CO(GT)
Candidate predictors,"Date, Time, NMHC(GT), C6H6(GT), NOx(GT), NO2(G..."


**Part 1 status:** data imported from the original repository, shape and columns checked, raw data saved as `data/AirQualityUCI.xlsx`, registration values recorded.

**Next (Part 2):** data dictionary, `-200` sentinel investigation, missing-value report, duplicate/timestamp/gap checks, and target distribution.